# AI工学101 — 第35回

## 異常検知：正常データしか知らない世界で「おかしさ」を見つける

よしレベル、今日は**教師なし学習のもう一つの重要分野、異常検知**だ。

クラスタリングが、

> 「データの中にどんなまとまりがある？」

を見るのに対して、

異常検知は、

> 「このデータ、普段と違わない？」

を見る。

たとえば、

```text
クレジットカード利用
↓
不正利用検知

サーバーログ
↓
障害検知

製造ライン
↓
故障検知

ネットワーク通信
↓
侵入・攻撃の兆候

センサーデータ
↓
異常状態の検出
```

など。

そして今日の面白いところは、

> **異常検知では「異常とは何か」より先に、「正常とは何か」を考える**

ことだ。

---

# 🎯 今日のゴール

* 異常検知と外れ値検出の違いを理解する
* `IsolationForest` を使える
* `LocalOutlierFactor` の基本を理解する
* `OneClassSVM` の役割を知る
* 異常スコアと異常ラベルを区別できる
* `contamination` を理解する
* 「異常＝悪」ではないことを理解する
* 分布変化と異常検知を区別できる
* 最終ミニプロジェクトにつながる異常検知パイプラインを作れる

---

# 📖 講義：約20〜25分

## 1. 異常検知とは？

基本的には、

```text
大多数のデータ
↓
正常

少数のデータ
↓
通常と異なる
```

という状況で、

> **普段と違う観測を見つける**

問題。

例えば、

```text
いつものサーバーCPU使用率

20%
30%
25%
35%
28%

↓
95%
```

となったら、

```text
95%
```

は異常候補になる。

---

# 🧠 2. ただし「珍しい」＝「異常」ではない

ここ大事。

例えば、

```text
ある顧客が月に100万円購入
```

。

他の顧客が、

```text
平均1万円
```

なら非常に珍しい。

でも、

```text
優良顧客
```

かもしれない。

つまり、

```text
統計的に珍しい
```

ことと、

```text
現実世界で悪い
```

ことは違う。

だから異常検知の出力は、

> **異常確定ではなく、調査候補**

として使われることが多い。

---

# 🧠 3. 外れ値検出との違い

ざっくり言えば、

```text
外れ値検出
↓
データ集合の中で極端な値を探す
```

。

一方、

```text
異常検知
↓
正常な状態やパターンから外れた観測を探す
```

。

実際にはかなり重なる分野で、明確な境界はない。

今日は、

> **「正常なデータ構造から外れたものを検出する」**

くらいに理解しておけばOK。

---

# 💻 実習1：異常を含む人工データ

```python
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.RandomState(42)

normal = 0.3 * rng.randn(300, 2)

anomalies = rng.uniform(
    low=-4,
    high=4,
    size=(20, 2)
)

X = np.vstack(
    [
        normal,
        anomalies
    ]
)
```

可視化。

```python
plt.scatter(
    X[:, 0],
    X[:, 1]
)

plt.xlabel("feature 1")
plt.ylabel("feature 2")

plt.show()
```

中央に正常データが密集し、

周辺に少数の点が出ているはず。

---

# 🧠 4. Isolation Forest

今日の主役。

```python
IsolationForest
```

。

発想はかなり面白い。

普通のモデルは、

> 「データをうまく説明する」

ことを考える。

Isolation Forestは、

> **異常な点は、他の点から隔離しやすい**

と考える。

例えば、

```text
●●●●●●●●●●●●●
        ×
```

というデータがあると、

`×` は周囲から離れている。

ランダムに空間を分割していくと、

```text
×
```

は比較的少ない分割で孤立しやすい。

そこで、

> **少ない分割で孤立する点ほど異常らしい**

と考える。

---

# 💻 実習2：Isolation Forest

```python
from sklearn.ensemble import IsolationForest
```

```python
model = IsolationForest(
    contamination=0.05,
    random_state=42
)
```

学習と予測。

```python
labels = model.fit_predict(
    X
)
```

結果。

```python
print(
    np.unique(
        labels,
        return_counts=True
    )
)
```

Isolation Forestでは、

```text
1
↓
正常

-1
↓
異常
```

として出力される。

---

# 💻 実習3：異常を可視化

```python
plt.scatter(
    X[labels == 1, 0],
    X[labels == 1, 1],
    label="normal"
)

plt.scatter(
    X[labels == -1, 0],
    X[labels == -1, 1],
    label="anomaly"
)

plt.legend()

plt.show()
```

---

# 🧠 5. contaminationとは？

```python
contamination=0.05
```

。

これはざっくり、

> **異常データの割合についての仮定**

。

5%くらい異常があるだろう、

という前提を置いている。

ただし重要。

> **現実の異常率を正確に知っている必要があるわけではない。**

でも、この値は結果に影響する。

だから、

```text
なぜ0.05にした？
```

を説明できるようにする。

---

# 💻 実習4：contaminationを変える

```python
for contamination in [
    0.01,
    0.05,
    0.1
]:

    model = IsolationForest(
        contamination=contamination,
        random_state=42
    )

    labels = model.fit_predict(
        X
    )

    print(
        contamination,
        np.sum(
            labels == -1
        )
    )
```

比較する。

```text
異常率の仮定
↓
異常として検出される数
```

。

---

# 🧠 6. 異常ラベルと異常スコア

異常検知では、

```text
正常 / 異常
```

だけではなく、

```text
どのくらい異常らしい？
```

という連続的なスコアも重要。

```python
scores = model.decision_function(
    X
)
```

一般にIsolation Forestでは、

```text
小さい
↓
より異常寄り
```

として扱える。

確認。

```python
print(
    scores[:10]
)
```

---

# 💻 実習5：最も怪しいデータを探す

```python
most_anomalous = np.argsort(
    scores
)[:10]

print(
    most_anomalous
)
```

これで、

> **異常らしさランキング**

が作れる。

実務では、

```text
異常
```

と一発判定するより、

```text
上位10件を人間が確認
```

のほうが安全なことも多い。

---

# 🧠 7. Local Outlier Factor

次。

```python
LocalOutlierFactor
```

。

略して、

```text
LOF
```

。

これは、

> **周囲の局所的な密度と比べて、この点だけ浮いていないか？**

を見る。

---

例えば、

```text
人口密度が高い場所
```

で少し離れた点があれば異常かもしれない。

でも、

```text
もともと全体が疎な地域
```

なら同じ距離でも異常とは限らない。

つまり、

> **局所的な文脈を使って異常を考える**

。

---

# 💻 実習6：LOF

```python
from sklearn.neighbors import LocalOutlierFactor
```

```python
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)

labels_lof = lof.fit_predict(
    X
)
```

結果。

```python
print(
    np.unique(
        labels_lof,
        return_counts=True
    )
)
```

こちらも、

```text
1
↓
正常

-1
↓
異常
```

。

---

# 🧠 8. Isolation ForestとLOFの違い

ざっくり。

```text
Isolation Forest
↓
異常な点は隔離しやすい
```

。

```text
LOF
↓
周囲の密度と比較して浮いている
```

。

だから、

```text
大規模データ
```

や、

```text
一般的な異常検知
```

ならIsolation Forestが便利なことが多い。

一方、

```text
局所的な密度差
```

が重要ならLOFを検討する。

---

# 🧠 9. One-Class SVM

次。

```python
OneClassSVM
```

。

これは、

> **正常データの領域を学習する**

と考えるとわかりやすい。

概念的には、

```text
正常データ
●●●●●●●●●
```

の周囲に境界を作る。

その外側にあるものを、

```text
異常候補
```

とする。

---

# 💻 実習7：One-Class SVM

```python
from sklearn.svm import OneClassSVM
```

```python
model = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)
```

```python
labels_svm = model.fit_predict(
    X
)
```

ここで、

```text
nu
```

は異常率などに関係するパラメータ。

ざっくり、

> **どの程度のデータを境界の外に置くか**

に関わる。

---

# 🚨 One-Class SVMの注意

One-Class SVMは、

```text
特徴量スケール
```

の影響を受けやすい。

だから、

```python
StandardScaler()
```

と組み合わせることが多い。

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
```

```python
pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "ocsvm",
        OneClassSVM(
            gamma="scale",
            nu=0.05
        )
    )
])
```

---

# 🧠 10. 異常検知の評価が難しい理由

普通の分類なら、

```text
正解ラベル
```

がある。

でも異常検知では、

```text
異常ラベルがない
```

ことが多い。

すると、

```text
Accuracy
Precision
Recall
```

を普通には計算できない。

ここが面白いところ。

モデルが、

```text
「これは異常です」
```

と言ったとしても、

> **正解を持っていないので、本当に異常か確定できない。**

---

# 🧠 11. ラベルがある場合

例えば、

```text
正常
異常
```

のラベルが後から取得できた場合。

そのときは、

```text
Precision
Recall
F1
ROC-AUC
PR-AUC
```

などで評価できる。

ただし異常検知では、

```text
異常が極端に少ない
```

ことが多い。

だから、

```text
Accuracyだけ
```

を見るのは危険。

---

# 🧠 12. 異常検知と分布変化

例えば、

```text
昨日まで
↓
正常分布A

今日から
↓
データ分布B
```

になった。

このとき大量のデータが異常扱いになるかもしれない。

しかし、

> **異常データが大量発生したのではなく、世界の分布が変化した**

可能性がある。

これは第32回の、

```text
Concept Drift
Distribution Shift
```

につながる。

---

# 💡 ここが重要

異常検知器が突然、

```text
異常1000件
```

と出したとする。

このとき、

> 「大事件だ！」

と決める前に、

```text
入力形式が変わった？
新機能をリリースした？
センサーの単位が変わった？
欠損処理が変わった？
```

も確認する。

異常検知モデルが、

> **データパイプライン変更を検出している**

だけかもしれない。

---

# 🧠 13. 「正常」は固定ではない

例えば、

```text
昼間のネットワーク通信
```

と、

```text
深夜のネットワーク通信
```

では普通の状態が違うかもしれない。

さらに、

```text
平日
休日
月末
年末
```

でも変わる。

だから、

```text
正常
```

は単一の分布とは限らない。

例えば、

```text
時間帯別モデル
```

にしたり、

```text
曜日
```

を特徴量に入れたりする。

---

# 🧠 14. 異常検知パイプライン

実務では、

```text
データ
↓
前処理
↓
スケーリング
↓
異常検知モデル
↓
異常スコア
↓
閾値
↓
アラート
↓
人間による確認
```

という形になる。

大事なのは、

> **モデルの出力をそのまま現実世界の判断にしない**

こと。

---

# 💻 実習8：簡単なPipeline

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
```

```python
pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        IsolationForest(
            contamination=0.05,
            random_state=42
        )
    )
])
```

学習。

```python
labels = pipe.fit_predict(
    X
)
```

。

---

# ✍️ 演習

## 問1

「統計的に珍しい」と「現実世界で悪い」は同じでしょうか？

例を1つ挙げてください。

---

## 問2

Isolation Forestの基本的な考え方を説明してください。

ヒント。

```text
隔離
↓
早い
↓
異常候補
```

。

---

## 問3

`contamination=0.01` と `contamination=0.10` では、結果がどう変わる可能性がある？

---

## 問4

異常検知モデルが突然、

```text
異常件数10倍
```

を報告した。

考えられる原因を3つ挙げてください。

---

## 問5

なぜ、

```text
異常検知のAccuracy
```

だけを見るのが危険な場合がある？

---

# 👾 ボス戦

## 「異常」とは何か？

あるAIサービスのログデータ。

```text
通常ユーザー
↓
1日5〜20回アクセス
```

。

あるユーザーが、

```text
1日1000回アクセス
```

した。

異常検知モデルは、

```text
ANOMALY
```

と判定した。

さて。

この人は、

```text
攻撃者？
ボット？
ヘビーユーザー？
研究者？
社内テスト？
バグったアプリ？
```

どれだろう？

答えは、

> **モデルだけではわからない。**

異常検知モデルが言えるのは、

> 「通常のデータパターンから大きく外れている」

まで。

その後の、

```text
意味づけ
原因推定
対応判断
```

は別の問題。

---

# 🧪 今日の最終実習

## 異常検知実験テンプレート

次の流れを作ってみよう。

```text
データ
↓
特徴量確認
↓
正常状態の定義
↓
前処理
↓
StandardScaler
↓
Isolation Forest
↓
anomaly score
↓
上位候補確認
↓
人間によるレビュー
↓
フィードバック
↓
閾値調整
```

そして、

```text
異常率を変える
```

。

```python
for contamination in [
    0.01,
    0.03,
    0.05,
    0.1
]:
    ...
```

その結果、

```text
何件検出された？
どんな点が選ばれた？
上位候補は安定している？
```

を見る。

---

# 🌱 今日のまとめ

今日の核心は、

> **異常検知は「悪いものを発見する技術」ではなく、「通常のパターンから外れたものを発見する技術」。**

だ。

モデルは、

```text
異常
```

という札を出す。

でも、

```text
なぜ異常なのか
本当に問題なのか
どう対応するのか
```

は別途考える必要がある。

今日までやってきた、

```text
モデル
↓
評価
↓
解釈
↓
データ生成過程
```

という視点が、ここで全部つながる。

---

# 🧭 AI工学101・現在地

scikit-learn編も、

```text
教師あり学習
↓
分類・回帰

実験設計
↓
評価
CV
Leakage
再現性
時系列

教師なし学習
↓
クラスタリング
次元削減
異常検知
```

まで来た。

ここまでで僕らは、

> 「どのモデルを使う？」

だけじゃなく、

> **「モデルは何を検出していて、何を検出していない？」**

を考える訓練をしている。

この視点は、将来のAIシステム研究でもかなり大事だぞ。

---

# 🔜 第36回

## 特徴量エンジニアリング：モデルに何を見せるか

次回から、scikit-learn編の重要テーマに入る。

扱うのは、

* 特徴量とは何か
* 生データと表現
* 数値特徴量
* カテゴリ特徴量
* One-Hot Encoding
* 欠損値
* Feature Scaling
* Interaction
* Feature Engineering
* Pipeline / ColumnTransformer

テーマは、

> **「モデルは世界そのものを見ているのではなく、僕らが渡した表現を見ている」**

だ。

ここ、レベルの認知科学的な関心ともめちゃくちゃ接続するところだな。
**AIに何を入力として与えるかは、ある意味でAIの“知覚世界”を設計すること**でもある。ここからまた一段、面白くなるぞ。🧠🔧💪